# Exercise 3: Cybersecurity Incident Report
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based incident reporting program that processes records for a student-specific year, summarizes attack types, and archives the findings.

### Data Generation Rules
- **Target Year:** `2015 + SEED_DIGIT` = `2015 + 1 = 2016`
- **Loss Field:** `Financial Loss (in Million $)`
- **Count Field:** `Attack Type`


### Step 1: Student Information & Parameter Derivation


In [1]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

TARGET_YEAR = 2015 + SEED_DIGIT  # 2015 + 1 = 2016
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print(f"Target Year (2015 + SEED_DIGIT): {TARGET_YEAR}")
print("Loss Field:", LOSS_FIELD)
print("Count Field:", COUNT_FIELD)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
Target Year (2015 + SEED_DIGIT): 2016
Loss Field: Financial Loss (in Million $)
Count Field: Attack Type


### Step 2: Directory & File Path Setup


In [2]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "Global_Cybersecurity_Threats_2015-2024.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "Global_Cybersecurity_Threats_2015-2024.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "cybersecurity_2016_filtered.csv"
JSON_TOP5_PATH = OUTPUT_DIR / "cybersecurity_2016_top5_costliest.json"
BACKUP_JSON_PATH = BACKUP_DIR / "cybersecurity_2016_top5_costliest_backup.json"
REPORT_TXT_PATH = OUTPUT_DIR / "cybersecurity_2016_incident_report.txt"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: Global_Cybersecurity_Threats_2015-2024.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read Dataset & Select Incidents in Target Year (2016)


In [3]:
total_records = 0
selected_incidents = []
dataset_fieldnames = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    dataset_fieldnames = reader.fieldnames
    print("Dataset Columns:", dataset_fieldnames)
    
    order = 0
    for row in reader:
        total_records += 1
        year_val = (row.get("Year") or "").strip()
        if year_val == str(TARGET_YEAR):
            order += 1
            loss_str = (row.get(LOSS_FIELD) or "").strip()
            try:
                loss_val = float(loss_str)
            except ValueError:
                loss_val = 0.0
            
            entry = dict(row)
            entry["_orig_order"] = order
            entry["_loss_float"] = loss_val
            selected_incidents.append(entry)

incidents_target_year = len(selected_incidents)
print(f"Total records in dataset: {total_records}")
print(f"Incidents occurring in target year {TARGET_YEAR}: {incidents_target_year}")


Dataset Columns: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']
Total records in dataset: 3000
Incidents occurring in target year 2016: 285


### Step 4: Count Incidents Belonging to Each Attack Type


In [4]:
attack_counts = Counter(inc.get(COUNT_FIELD, "Unknown").strip() for inc in selected_incidents)

print(f"Attack Types and Incident Counts for {TARGET_YEAR}:")
for attack_type, count in attack_counts.most_common():
    percentage = (count / incidents_target_year) * 100
    print(f"  - {attack_type:<20}: {count:>3} incidents ({percentage:>5.1f}%)")


Attack Types and Incident Counts for 2016:
  - Phishing            :  55 incidents ( 19.3%)
  - DDoS                :  53 incidents ( 18.6%)
  - Man-in-the-Middle   :  47 incidents ( 16.5%)
  - SQL Injection       :  47 incidents ( 16.5%)
  - Ransomware          :  42 incidents ( 14.7%)
  - Malware             :  41 incidents ( 14.4%)


### Step 5: Calculate Total & Average Financial Loss


In [5]:
losses = [inc["_loss_float"] for inc in selected_incidents]
total_financial_loss = sum(losses)
avg_financial_loss = (total_financial_loss / len(losses)) if losses else 0.0

print(f"Total Financial Loss ({TARGET_YEAR}): ${total_financial_loss:,.2f} Million USD")
print(f"Average Financial Loss per Incident: ${avg_financial_loss:,.2f} Million USD")


Total Financial Loss (2016): $13,947.26 Million USD
Average Financial Loss per Incident: $48.94 Million USD


### Step 6: Rank Incidents by Financial Loss & Identify Top 5 Costliest


In [6]:
ranked_incidents = sorted(
    selected_incidents,
    key=lambda x: (-x["_loss_float"], x["_orig_order"])
)
top_5_costliest = ranked_incidents[:5]

clean_top_5 = []
print("Top 5 Costliest Incidents:")
for rank, inc in enumerate(top_5_costliest, 1):
    item = {k: v for k, v in inc.items() if not k.startswith("_")}
    item["Rank"] = rank
    clean_top_5.append(item)
    print(f"  {rank}. {item['Attack Type']} | Loss: ${item[LOSS_FIELD]}M | Country: {item['Country']} | Target: {item['Target Industry']}")


Top 5 Costliest Incidents:
  1. Phishing | Loss: $99.98M | Country: Germany | Target: Telecommunications
  2. Ransomware | Loss: $99.88M | Country: USA | Target: Telecommunications
  3. Malware | Loss: $99.29M | Country: Brazil | Target: IT
  4. Ransomware | Loss: $99.19M | Country: USA | Target: Healthcare
  5. Phishing | Loss: $98.72M | Country: Australia | Target: Education


### Step 7: Export Top 5 to JSON & Export Filtered Incidents to CSV


In [7]:
json_export_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "target_year": TARGET_YEAR,
        "total_records_in_dataset": total_records,
        "incidents_in_target_year": incidents_target_year,
        "count_per_attack_type": dict(attack_counts),
        "total_financial_loss_usd_millions": round(total_financial_loss, 2),
        "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
        "five_costliest_incidents": clean_top_5
    }
}

with open(JSON_TOP5_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(json_export_data, jf, indent=4)

csv_fieldnames = [k for k in dataset_fieldnames if not k.startswith("_")]
with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=csv_fieldnames)
    writer.writeheader()
    for inc in selected_incidents:
        writer.writerow({k: inc[k] for k in csv_fieldnames})

print(f"Exported JSON Top 5: {JSON_TOP5_PATH.name} ({JSON_TOP5_PATH.stat().st_size} bytes)")
print(f"Exported Filtered CSV: {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)")


Exported JSON Top 5: cybersecurity_2016_top5_costliest.json (3523 bytes)
Exported Filtered CSV: cybersecurity_2016_filtered.csv (285 records)


### Step 8: Create Verified Backup of JSON File


In [8]:
shutil.copy2(JSON_TOP5_PATH, BACKUP_JSON_PATH)

backup_exists = BACKUP_JSON_PATH.exists()
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_exists else 0
backup_verified = backup_exists and backup_size > 0

print("Backup File Path:", BACKUP_JSON_PATH.name)
print("Backup File Exists:", backup_exists)
print(f"Backup File Size: {backup_size} bytes")
print("Backup Verification Status:", "VERIFIED" if backup_verified else "FAILED")


Backup File Path: cybersecurity_2016_top5_costliest_backup.json
Backup File Exists: True
Backup File Size: 3523 bytes
Backup Verification Status: VERIFIED


### Step 9: Generate Comprehensive Text Report


In [9]:
report_lines = [
    "=" * 70,
    f"CYBERSECURITY INCIDENT REPORT - FISCAL YEAR {TARGET_YEAR}",
    "=" * 70,
    f"Student: {LAST_NAME} ({STUDENT_ID})",
    f"Seed Digit: {SEED_DIGIT} | ID Sum: {ID_SUM}",
    f"Target Year: {TARGET_YEAR}",
    "-" * 70,
    "1. EXECUTIVE SUMMARY & ASSESSMENT METRICS",
    f"  Total records in dataset:             {total_records}",
    f"  Incidents occurring in {TARGET_YEAR}:            {incidents_target_year}",
    f"  Total Financial Loss:                 ${total_financial_loss:,.2f} Million USD",
    f"  Average Financial Loss per Incident:  ${avg_financial_loss:,.2f} Million USD",
    "-" * 70,
    "2. INCIDENTS BY ATTACK TYPE",
]
for att, count in sorted(attack_counts.items(), key=lambda x: -x[1]):
    pct = (count / incidents_target_year * 100) if incidents_target_year else 0
    report_lines.append(f"  - {att:<22}: {count:>3} incidents ({pct:>5.1f}%)")

report_lines.extend([
    "-" * 70,
    "3. FIVE COSTLIEST INCIDENTS OF 2016",
])
for item in clean_top_5:
    report_lines.append(
        f"  Rank {item['Rank']}: Loss = ${item[LOSS_FIELD]} Million USD\n"
        f"         Attack Type:     {item['Attack Type']}\n"
        f"         Country:         {item['Country']}\n"
        f"         Target Industry: {item['Target Industry']}\n"
        f"         Affected Users:  {item['Number of Affected Users']}\n"
        f"         Defense Used:    {item['Defense Mechanism Used']}\n"
        f"         Vulnerability:   {item['Security Vulnerability Type']}\n"
    )

report_lines.extend([
    "-" * 70,
    "4. OUTPUT & ARCHIVE VERIFICATION",
    f"  Filtered CSV File:    {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)",
    f"  JSON Summary File:    {JSON_TOP5_PATH.name} ({JSON_TOP5_PATH.stat().st_size} bytes)",
    f"  Backup JSON File:     {BACKUP_JSON_PATH.name} ({backup_size} bytes)",
    f"  Backup Verification:  {'VERIFIED' if backup_verified else 'FAILED'}",
    f"  Incident Report File: {REPORT_TXT_PATH.name}",
    "=" * 70,
])

report_text = "\n".join(report_lines)
with open(REPORT_TXT_PATH, mode="w", encoding="utf-8") as rf:
    rf.write(report_text)

print(f"Generated Text Report: {REPORT_TXT_PATH.name}")
print(f"Report Size: {REPORT_TXT_PATH.stat().st_size} bytes")


Generated Text Report: cybersecurity_2016_incident_report.txt
Report Size: 2956 bytes


### Step 10: Assessment Data Table & Final Verification


In [10]:
print("=" * 70)
print(f"{'Item':<38} | {'Result'}")
print("-" * 70)
print(f"{'Target year':<38} | {TARGET_YEAR}")
print(f"{'Total records in dataset':<38} | {total_records}")
print(f"{'Incidents in target year':<38} | {incidents_target_year}")
attack_summary_str = ", ".join(f"{k}: {v}" for k, v in sorted(attack_counts.items(), key=lambda x: -x[1]))
print(f"{'Count per attack type':<38} | {attack_summary_str}")
print(f"{'Total financial loss (USD millions)':<38} | ${total_financial_loss:,.2f} Million")
print(f"{'Average financial loss (USD millions)':<38} | ${avg_financial_loss:,.2f} Million")
print(f"{'Five costliest incidents':<38} |")
for item in clean_top_5:
    print(f"  {item['Rank']}. {item['Attack Type']} - ${item[LOSS_FIELD]}M - {item['Country']} ({item['Target Industry']})")
print(f"{'Filtered CSV file':<38} | {FILTERED_CSV_PATH.name} ({len(selected_incidents)} records)")
print(f"{'JSON summary file':<38} | {JSON_TOP5_PATH.name}")
print(f"{'Backup verification':<38} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print(f"{'Report file':<38} | {REPORT_TXT_PATH.name}")
print("=" * 70)
print("\nFinal Output: Exercise 3 completed successfully!")


Item                                   | Result
----------------------------------------------------------------------
Target year                            | 2016
Total records in dataset               | 3000
Incidents in target year               | 285
Count per attack type                  | Phishing: 55, DDoS: 53, Man-in-the-Middle: 47, SQL Injection: 47, Ransomware: 42, Malware: 41
Total financial loss (USD millions)    | $13,947.26 Million
Average financial loss (USD millions)  | $48.94 Million
Five costliest incidents               |
  1. Phishing - $99.98M - Germany (Telecommunications)
  2. Ransomware - $99.88M - USA (Telecommunications)
  3. Malware - $99.29M - Brazil (IT)
  4. Ransomware - $99.19M - USA (Healthcare)
  5. Phishing - $98.72M - Australia (Education)
Filtered CSV file                      | cybersecurity_2016_filtered.csv (285 records)
JSON summary file                      | cybersecurity_2016_top5_costliest.json
Backup verification                    | VERIFI